# Phase 7: Multi-Output Training Dataset Preparation

## 🎯 Objective
Prepare ML-ready supervised datasets for multi-output 72-hour AQI forecasting ($X_t \rightarrow [AQI(t+1), \dots, AQI(t+72)]$).

### Key Steps:
1. **Accounting Audit**: Document row transformations from raw observations (49,483) to engineered features (48,808) and 72-hour supervised pairs (48,736).
2. **72-Hour Target Matrix Construction**: Build aligned multi-step target matrix $Y = [y_{t+1}, \dots, y_{t+72}]$ dropping broken/incomplete horizons.
3. **Chronological Splitting & 72h Embargo Gap**: Enforce 80/20 train/test split with a 72-hour embargo gap to eliminate target leakage across partitions.
4. **Train-Only Feature Scaling**: Fit `StandardScaler` exclusively on $X_{\text{train}}$ and serialize to `data/models/feature_scaler.joblib`.
5. **Artifact Verification**: Validate saved arrays (`X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy`) and timestamps.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.feature_pipeline.feature_engineering import FeatureEngineeringPipeline
from src.training_pipeline.dataset_builder import DatasetBuilder

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "data" / "models"

## 1. Dataset Row Accounting Audit

| Stage | Row Count | Change | Reason |
| :--- | :--- | :--- | :--- |
| **Raw API Ingestion** | **49,483** | Baseline | Historical records Nov 2020 - Aug 2026 |
| **Hourly Grid Reindexing** | **50,468** | +985 | Continuous 1h timeline grid |
| **Feature Engineering Output** | **48,808** | -1,660 | 24h warm-up + gap boundary lag drops |
| **72h Supervised Pairs** | **48,736** | -72 | Dropped tail end where $t+72$ is in the future |
| **Train Set ($X_{\text{train}}$)** | **38,917** | 80% | Chronological training partition |
| **Embargo Gap** | **71** | -71 | Samples purged at split to prevent target overlap |
| **Test Set ($X_{\text{test}}$)** | **9,748** | 20% | Out-of-time evaluation partition |

## 2. Execute Dataset Builder & Inspect Artifacts

In [ ]:
clean_csv = PROCESSED_DIR / "historical_aqi_clean.csv"
df_clean = pd.read_csv(clean_csv)

pipeline = FeatureEngineeringPipeline()
df_features, feature_names = pipeline.build_features(df_clean, drop_na=True)

builder = DatasetBuilder(forecast_horizons=72, train_ratio=0.8, target_col="epa_aqi")
summary = builder.build_and_save_dataset(df_features, feature_names)

print("\nDataset Build Summary:")
print(json.dumps(summary, indent=2))

## 3. Validate Train/Test Anti-Leakage Partitioning

In [ ]:
X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
X_test = np.load(PROCESSED_DIR / "X_test.npy")
y_test = np.load(PROCESSED_DIR / "y_test.npy")

df_train_times = pd.read_csv(PROCESSED_DIR / "train_timestamps.csv")
df_test_times = pd.read_csv(PROCESSED_DIR / "test_timestamps.csv")

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

t_train_max = pd.to_datetime(df_train_times['datetime_utc'].max())
t_test_min = pd.to_datetime(df_test_times['datetime_utc'].min())
gap_hours = (t_test_min - t_train_max) / pd.Timedelta(hours=1)

print(f"\nTrain Period End:    {t_train_max}")
print(f"Test Period Start:   {t_test_min}")
print(f"Embargo Gap:         {gap_hours:.1f} hours (Required: >= 72h)")
assert gap_hours >= 72, "ERROR: Embargo gap violated!"

## 4. Verify Feature Scaler Fitting

In [ ]:
scaler = joblib.load(MODELS_DIR / "feature_scaler.joblib")

print(f"Scaler n_features_in_: {scaler.n_features_in_}")
print(f"X_train mean across features (should be ~0): {X_train.mean():.6f}")
print(f"X_train std across features (should be ~1):  {X_train.std():.6f}")
print(f"X_test mean (uncentered test distribution):  {X_test.mean():.4f}")

## 5. Visualizing Target Trajectories Across Horizons

In [ ]:
plt.figure(figsize=(12, 5))
horizons = np.arange(1, 73)
for i in [100, 500, 1000, 2500]:
    plt.plot(horizons, y_train[i], label=f"Sample {i} ({df_train_times.iloc[i]['datetime_utc'][:10]})")

plt.title("Sample 72-Hour Future Target Trajectories [y_{t+1}, ..., y_{t+72}]")
plt.xlabel("Forecast Horizon Step (h=1 to h=72)")
plt.ylabel("US EPA AQI Target")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Conclusions for Phase 8 (Model Training: Ridge Regression)
- 38,917 training samples and 9,748 test samples prepared with 64 scaled features.
- 72 target steps per sample ready for direct multi-output models (`MultiOutputRegressor`).
- Strict embargo boundary guarantees zero test leakage.